In [2]:
import pandas as pd

# 1. Leer el archivo CSV (cambia 'tu_archivo.csv' por el nombre real de tu archivo)
df = pd.read_csv('../results/reports/predictions_discovery_reports/Top_100_Vaccine_Candidates.csv')

# 2. Abrir (o crear) un archivo con extensión .fasta en modo escritura ('w')
with open('../results/reports/predictions_discovery_reports/neoepitepos_filtrados.fasta', 'w') as fasta_file:
    
    # 3. Iterar sobre la columna "Peptide"
    # Usamos enumerate con start=1 para que el contador empiece en 1, 2, 3...
    for index, peptido in enumerate(df['Peptide'], start=1):
        
        # Escribimos el encabezado FASTA (>Neoantigeno_X)
        fasta_file.write(f'>Neoantigeno{index}\n')
        
        # Escribimos la secuencia del péptido justo debajo
        fasta_file.write(f'{peptido}\n')

print("¡Archivo FASTA generado con éxito! Revisa tu carpeta.")

¡Archivo FASTA generado con éxito! Revisa tu carpeta.


In [5]:
import pandas as pd
import requests
import time

def obtener_secuencia_uniprot(gen):
    """
    Se conecta a UniProt y descarga la secuencia completa de la proteína humana (revisada) para un gen dado.
    """
    # Buscamos el gen exacto en humanos (organism_id:9606) y que esté revisado (Swiss-Prot)
    url = f"https://rest.uniprot.org/uniprotkb/search?query=gene_exact:{gen}+AND+organism_id:9606+AND+reviewed:true&format=fasta"
    
    try:
        response = requests.get(url)
        if response.status_code == 200 and response.text:
            # El resultado es un FASTA. Ignoramos la primera línea (>) y unimos el resto
            lineas = response.text.strip().split('\n')
            secuencia_completa = "".join([linea for linea in lineas if not linea.startswith(">")])
            return secuencia_completa
    except:
        pass
    return None

def generar_slp_magico(peptido_mutado, peptido_wt, gen, target_length=18):
    """
    Descarga la proteína, busca el péptido WT y devuelve el fragmento de 18aa con la mutación.
    """
    pep_len = len(peptido_mutado)
    
    # 1. Descargar la secuencia completa de internet
    secuencia_completa = obtener_secuencia_uniprot(gen)
    
    # Si no encuentra el gen en UniProt, devolvemos el péptido corto para no dar error
    if not secuencia_completa:
        return peptido_mutado
        
    # 2. Buscar el péptido wildtype dentro de la proteína gigante
    idx = secuencia_completa.find(str(peptido_wt))
    
    if idx != -1:
        # ¡Lo encontró! Calculamos los flancos
        missing_aa = target_length - pep_len
        if missing_aa <= 0:
            return peptido_mutado
            
        left_flank_len = missing_aa // 2
        right_flank_len = missing_aa - left_flank_len
        
        start_idx = max(0, idx - left_flank_len)
        end_idx = min(len(secuencia_completa), idx + pep_len + right_flank_len)
        
        # Extraemos flancos de la secuencia normal y ponemos el mutado en el medio
        left_flank = secuencia_completa[start_idx:idx]
        right_flank = secuencia_completa[idx + len(str(peptido_wt)):end_idx]
        
        return left_flank + str(peptido_mutado) + right_flank
        
    return peptido_mutado

# =====================================================================
# EJECUCIÓN PRINCIPAL
# =====================================================================

print("Leyendo CSV y conectando a UniProt... (Esto puede tardar un par de minutos para 100 genes)")

# 1. Leer el archivo CSV
df = pd.read_csv('../results/reports/predictions_discovery_reports/Top_100_Vaccine_Candidates.csv')

# 2. Abrir archivo FASTA de salida
ruta_salida = '../results/reports/predictions_discovery_reports/neoepitepos_SLP_18aa_uniprot.fasta'

# Variables para ver cómo va
exitos = 0

with open(ruta_salida, 'w') as fasta_file:
    for index, row in df.iterrows():
        peptido_mutado = row['Peptide']
        peptido_wt = row['wt_seq']
        gen = row['gene']
        
        # Generar secuencia conectándose a UniProt
        slp = generar_slp_magico(peptido_mutado, peptido_wt, gen, target_length=18)
        
        if len(slp) >= 18: exitos += 1
        
        # Escribimos en el FASTA
        fasta_file.write(f'>Neoantigeno{index+1}_{gen}\n')
        fasta_file.write(f'{slp}\n')
        
        # Pequeña pausa para no saturar el servidor de UniProt y que no nos bloqueen la conexión
        time.sleep(0.2)

print(f"¡Proceso terminado! Archivo guardado en {ruta_salida}.")
print(f"Se logró expandir con éxito a 18 aminoácidos {exitos} de {len(df)} neoantígenos.")

Leyendo CSV y conectando a UniProt... (Esto puede tardar un par de minutos para 100 genes)
¡Proceso terminado! Archivo guardado en ../results/reports/predictions_discovery_reports/neoepitepos_SLP_18aa_uniprot.fasta.
Se logró expandir con éxito a 18 aminoácidos 90 de 100 neoantígenos.
